# Volume 2, Chapter 7: Modules over a PID (Python companion)

Exercises first, solutions at the end. We implement the Smith normal form, count abelian groups, compute invariant factors and Jordan forms, count sublattices, and end with an application that goes from algebra to topology and data analysis: **computing homology with the Smith normal form**, which detects that a Möbius-like surface, the projective plane, has a loop that becomes trivial only when traversed twice.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import random
from collections import Counter
import sympy
from sympy import Matrix, symbols, QQ, ZZ, eye
from sympy.matrices.normalforms import smith_normal_form, invariant_factors
from sympy.utilities.iterables import partitions

## Exercises

**P7.1 (★) Smith normal form from scratch.** Implement the algorithm of Theorem 7.4 over $\Z$ with Euclidean divisions, returning $D$ and invertible $P,Q$ with $PAQ=D$. Check it on the matrix of Figure 7.1 and on random matrices against SymPy, and check $PAQ=D$, $\det P,\det Q=\pm1$.

In [ ]:
# Your code here


**P7.2 (★) Counting abelian groups, and $(\Z/n\Z)^*$.** Using Corollary 7.7, count the abelian groups of order $n$ for $n\le 2000$; which $n$ has the most? Then, for $n\le 200$, compute the invariant factors of $(\Z/n\Z)^*$ from Exercise 7.3, and check them by brute force: an abelian group $\bigoplus\Z/d_i$ has exactly $\prod_i\gcd(k,d_i)$ elements with $x^k=1$.

In [ ]:
# Your code here


**P7.3 (★★) Invariant factors and Jordan forms.** For a few integer matrices, compute the invariant factors of $XI-A$ over $\Q[X]$, the minimal and characteristic polynomials, and the Jordan form; check Cayley–Hamilton exactly. Then find two matrices with the same characteristic and minimal polynomials that are not similar (Exercise 7.5 suggests where to look).

In [ ]:
# Your code here


**P7.4 (★★) Sublattices.** Count the subgroups of index $n$ of $\Z^2$ by brute force for $n\le 12$ (they all contain $n\Z^2$, so they are the subgroups of order $n$ of $(\Z/n\Z)^2$), and compare with $\sigma(n)$ (Exercise 7.8).

In [ ]:
# Your code here


**P7.5 (★★★) Homology with the Smith normal form.** A triangulated surface gives free $\Z$-modules $C_0,C_1,C_2$ (formal combinations of vertices, edges, triangles) and boundary maps $\partial_2:C_2\to C_1$, $\partial_1:C_1\to C_0$ with $\partial_1\partial_2=0$; for instance $\partial[a,b,c]=[b,c]-[a,c]+[a,b]$. The homology groups $H_k=\ker\partial_k/\operatorname{im}\partial_{k+1}$ are finitely generated abelian groups: their rank counts $k$-dimensional holes, and their torsion records twisting. By Theorem 7.6 and Corollary 7.8, $H_k\cong\Z^{\dim C_k-\operatorname{rk}\partial_k-\operatorname{rk}\partial_{k+1}}\oplus\bigoplus\Z/d_i$, with $d_i>1$ the invariant factors of $\partial_{k+1}$.

Compute $H_0,H_1,H_2$ for the sphere (boundary of a tetrahedron), the torus (Möbius' $7$-vertex triangulation, triangles $\{i,i+1,i+3\}$ and $\{i,i+2,i+3\}$ mod $7$), the projective plane (the $10$ triangles of the hemi-icosahedron) and a Klein bottle (a $4\times4$ grid glued with a twist).

In [ ]:
# Your code here


---
## Solutions

### P7.1

In [ ]:
def smith(A):
    A = [list(map(int, r)) for r in A]
    m, n = len(A), len(A[0])
    P = [[int(i == j) for j in range(m)] for i in range(m)]
    Q = [[int(i == j) for j in range(n)] for i in range(n)]
    def row_op(i, j, c):        # R_i += c R_j
        A[i] = [a + c * b for a, b in zip(A[i], A[j])]; P[i] = [a + c * b for a, b in zip(P[i], P[j])]
    def col_op(i, j, c):        # C_i += c C_j
        for M in (A, Q):
            for r in M:
                r[i] += c * r[j]
    def swap_rows(i, j):
        A[i], A[j] = A[j], A[i]; P[i], P[j] = P[j], P[i]
    def swap_cols(i, j):
        for M in (A, Q):
            for r in M:
                r[i], r[j] = r[j], r[i]
    for t in range(min(m, n)):
        while True:
            nz = [(abs(A[i][j]), i, j) for i in range(t, m) for j in range(t, n) if A[i][j]]
            if not nz:
                return A, P, Q
            _, i, j = min(nz)
            swap_rows(t, i); swap_cols(t, j)
            clean = True
            for i in range(t + 1, m):
                row_op(i, t, -(A[i][t] // A[t][t]))
                clean &= A[i][t] == 0
            for j in range(t + 1, n):
                col_op(j, t, -(A[t][j] // A[t][t]))
                clean &= A[t][j] == 0
            if not clean:
                continue                       # a smaller remainder appeared: new pivot
            bad = [i for i in range(t + 1, m) for j in range(t + 1, n) if A[i][j] % A[t][t]]
            if bad:
                row_op(t, bad[0], 1); continue
            if A[t][t] < 0:
                A[t] = [-a for a in A[t]]; P[t] = [-a for a in P[t]]
            break
    return A, P, Q

A = [[2, 4, 4], [-6, 6, 12], [10, -4, -16]]
D, P, Q = smith(A)
print(D, Matrix(P) * Matrix(A) * Matrix(Q) == Matrix(D), Matrix(P).det(), Matrix(Q).det())
random.seed(7)
ok = True
for _ in range(200):
    m, n = random.randint(1, 5), random.randint(1, 5)
    B = [[random.randint(-20, 20) for _ in range(n)] for _ in range(m)]
    D, P, Q = smith(B)
    mine = [D[i][i] for i in range(min(m, n)) if D[i][i]]
    ref = [abs(x) for x in invariant_factors(Matrix(B), domain=ZZ) if x != 0]
    ok &= mine == ref and Matrix(P) * Matrix(B) * Matrix(Q) == Matrix(D) and abs(Matrix(P).det()) == 1 == abs(Matrix(Q).det())
print('200 random matrices agree with SymPy:', ok)

The figure's matrix gives $\operatorname{diag}(2,6,12)$, with $\det P=\pm1$, $\det Q=\pm1$. Each pass picks the entry of smallest absolute value as pivot and divides: remainders are smaller than the pivot, so the pivot shrinks until it divides its row and column. This is the Euclidean version of the gcd step of the proof. On large matrices this naive algorithm lets the entries of $P$ and $Q$ explode, which is why serious implementations work modulo the determinant (Kannan and Bachem, in the sources).

### P7.2

In [ ]:
def num_abelian(n):
    return math.prod(int(sympy.partition(a)) for a in sympy.factorint(n).values())

counts = {n: num_abelian(n) for n in range(1, 2001)}
best = max(counts, key=counts.get)
print('360:', counts[360], '  most groups for n <= 2000:', best, sympy.factorint(best), counts[best])

def units_invariants(n):
    # elementary divisors from Volume 1 (odd p^k cyclic) and Exercise 7.3 (2^k), then recombine
    prime_powers = []
    for p, k in sympy.factorint(n).items():
        if p == 2:
            if k == 2: prime_powers.append(2)
            elif k >= 3: prime_powers += [2, 2 ** (k - 2)]
        else:
            m = (p - 1) * p ** (k - 1)
            prime_powers += [q ** e for q, e in sympy.factorint(m).items()]
    by_prime = {}
    for x in prime_powers:
        q = sympy.primefactors(x)[0]
        by_prime.setdefault(q, []).append(x)
    r = max((len(v) for v in by_prime.values()), default=0)
    inv = [1] * r
    for q, v in by_prime.items():
        v = sorted(v, reverse=True)
        for i, x in enumerate(v):
            inv[r - 1 - i] *= x
    return inv

ok = True
for n in range(2, 201):
    inv = units_invariants(n)
    U = [a for a in range(1, n) if math.gcd(a, n) == 1]
    for k in range(1, 25):
        ok &= sum(1 for a in U if pow(a, k, n) == 1) == math.prod(math.gcd(k, d) for d in inv)
print('invariant factors of (Z/n)^* confirmed for n <= 200:', ok)
print('(Z/360)^* =', ' + '.join(f'Z/{d}' for d in units_invariants(360)))
print('cyclic for n <= 50:', [n for n in range(2, 51) if len(units_invariants(n)) <= 1])

There are $6$ abelian groups of order $360$. Up to $2000$ the record is held by $1024=2^{10}$, with $\mathrm p(10)=42$ abelian groups: high powers of a single prime win. (Compare with the $49\,487\,365\,422$ groups of order $1024$ in all, Chapter 5.) $(\Z/360\Z)^*\cong\Z/2\oplus\Z/2\oplus\Z/2\oplus\Z/12$ as in Exercise 7.3, and the brute-force counts of solutions of $x^k=1$ confirm the invariant factors for every $n\le200$ (these counts determine a finite abelian group). The cyclic cases are $n=2,4,p^k,2p^k$, Volume 1's theorem.

### P7.3

In [ ]:
x = symbols('x')
def report(A):
    A = Matrix(A); n = A.shape[0]
    inv = [f for f in invariant_factors(x * eye(n) - A, domain=QQ[x]) if sympy.degree(f, x) > 0]
    chi = A.charpoly(x).as_expr()
    mu = sympy.factor(inv[-1])
    J = A.jordan_form()[1]
    CH = sympy.zeros(n)
    for k, c in enumerate(reversed(sympy.Poly(chi, x).all_coeffs())):
        CH += c * A**k
    print('invariant factors:', [sympy.factor(f) for f in inv], '  chi:', sympy.factor(chi), '  mu:', mu, '  chi(A)=0:', CH == sympy.zeros(n))
    return J

print(report([[2, 1, 0], [0, 2, 0], [0, 0, 2]]))
print(report([[0, -1], [1, 0]]))
print(report([[1, 1, 0, 0], [-1, 3, 0, 0], [0, 0, 2, 1], [0, 0, 0, 2]]))
N1 = Matrix([[0, 1, 0, 0], [0, 0, 0, 0], [0, 0, 0, 1], [0, 0, 0, 0]])     # blocks 2 + 2
N2 = Matrix([[0, 1, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]])     # blocks 2 + 1 + 1
for N in (N1, N2):
    print('nilpotent: mu = x^2 ?', N**2 == sympy.zeros(4) and N != sympy.zeros(4), ' chi =', N.charpoly(x).as_expr(), ' rank =', N.rank(),
          ' invariant factors:', [f for f in invariant_factors(x * eye(4) - N, domain=QQ[x]) if sympy.degree(f, x) > 0])

The first matrix has invariant factors $X-2,(X-2)^2$: $\mu=(X-2)^2\ne\chi=(X-2)^3$, and its Jordan form has blocks of sizes $1$ and $2$. The third matrix, made of two $2\times2$ blocks with the single eigenvalue $2$, has invariant factors $(X-2)^2,(X-2)^2$ and two Jordan blocks of size $2$. The rotation by $90°$ has the single invariant factor $X^2+1$, irreducible over $\Q$: no Jordan form over $\Q$, but its rational canonical form is the companion matrix of $X^2+1$, which it already is. In every case $\chi_A(A)=0$ exactly.

The two nilpotent $4\times4$ matrices with blocks $2+2$ and $2+1+1$ both have $\chi=X^4$ and $\mu=X^2$, but ranks $2$ and $1$: they are not similar. Their invariant factors, $(X^2,X^2)$ and $(X,X,X^2)$, tell them apart. In size $\le3$ the pair $(\chi,\mu)$ always determines the class; from size $4$ on it does not.

### P7.4

In [ ]:
def subgroups_of_order(n):
    elems = [(a, b) for a in range(n) for b in range(n)]
    found = set()
    for g, h in itertools.combinations_with_replacement(elems, 2):
        S = {((i * g[0] + j * h[0]) % n, (i * g[1] + j * h[1]) % n) for i in range(n) for j in range(n)}
        if len(S) == n:
            found.add(frozenset(S))
    return len(found)

print([(n, subgroups_of_order(n), sympy.divisor_sigma(n)) for n in range(1, 13)])

The brute-force counts equal $\sigma(n)$: $1,3,4,7,6,12,8,15,13,18,12,28$. Every subgroup of $(\Z/n\Z)^2$ is generated by two elements (it is a quotient of a subgroup of $\Z^2$, free of rank $\le2$ by Lemma 7.5), so pairs of generators suffice. The generating function $\sum\sigma(n)n^{-s}=\zeta(s)\zeta(s-1)$ is the first example of a *subgroup zeta function*.

### P7.5

In [ ]:
def boundary_matrices(triangles):
    tris = sorted({tuple(sorted(t)) for t in triangles})
    edges = sorted({e for t in tris for e in itertools.combinations(t, 2)})
    verts = sorted({v for t in tris for v in t})
    ei = {e: i for i, e in enumerate(edges)}; vi = {v: i for i, v in enumerate(verts)}
    d2 = [[0] * len(tris) for _ in edges]
    for j, (a, b, c) in enumerate(tris):
        d2[ei[(b, c)]][j] += 1; d2[ei[(a, c)]][j] -= 1; d2[ei[(a, b)]][j] += 1
    d1 = [[0] * len(edges) for _ in verts]
    for j, (a, b) in enumerate(edges):
        d1[vi[b]][j] += 1; d1[vi[a]][j] -= 1
    return d1, d2, (len(verts), len(edges), len(tris))

def homology(triangles):
    d1, d2, (c0, c1, c2) = boundary_matrices(triangles)
    assert all(v == 0 for v in (Matrix(d1) * Matrix(d2)))          # boundary of a boundary is 0
    D1 = [smith(d1)[0][i][i] for i in range(min(len(d1), len(d1[0])))]
    D2 = [smith(d2)[0][i][i] for i in range(min(len(d2), len(d2[0])))]
    r1 = sum(1 for d in D1 if d); r2 = sum(1 for d in D2 if d)
    tors1 = [abs(d) for d in D2 if abs(d) > 1]
    tors0 = [abs(d) for d in D1 if abs(d) > 1]
    fmt = lambda rank, tors: ' + '.join((['Z^%d' % rank] if rank else []) + ['Z/%d' % t for t in tors]) or '0'
    return fmt(c0 - r1, tors0), fmt(c1 - r1 - r2, tors1), fmt(c2 - r2, []), c0 - c1 + c2

sphere = list(itertools.combinations(range(4), 3))
torus = [((i) % 7, (i + 1) % 7, (i + 3) % 7) for i in range(7)] + [((i) % 7, (i + 2) % 7, (i + 3) % 7) for i in range(7)]
rp2 = [(0, 1, 2), (0, 2, 3), (0, 3, 4), (0, 4, 5), (0, 5, 1), (1, 2, 4), (2, 3, 5), (3, 4, 1), (4, 5, 2), (5, 1, 3)]
def grid_surface(n, twist):
    def v(i, j):
        if i == n:
            i, j = 0, ((n - j) % n if twist else j)
        return i * n + (j % n)
    tris = []
    for i in range(n):
        for j in range(n):
            a, b, c, d = v(i, j), v(i + 1, j), v(i, j + 1), v(i + 1, j + 1)
            tris += [(a, b, d), (a, c, d)]
    return tris
for name, T in [('sphere', sphere), ('torus', torus), ('projective plane', rp2),
                ('torus (grid)', grid_surface(4, False)), ('Klein bottle (grid)', grid_surface(4, True))]:
    H0, H1, H2, chi = homology(T)
    print(f'{name:20s} H0 = {H0:5s} H1 = {H1:10s} H2 = {H2:4s} Euler characteristic {chi}')

The results: the sphere has $H_1=0$ and $H_2=\Z$ (one enclosed cavity); the torus has $H_1=\Z^2$ (two independent loops) and $H_2=\Z$; the projective plane has $H_1=\Z/2$ and $H_2=0$; the Klein bottle has $H_1=\Z\oplus\Z/2$ and $H_2=0$. The torsion $\Z/2$ is a loop that does not bound anything, but whose double does: it comes from an invariant factor $2$ of $\partial_2$, and it is invisible over $\Q$, where only ranks survive. The Euler characteristics $V-E+F=2,0,1,0$ also follow from the ranks.

This is the algebra behind *topological data analysis*: from a cloud of data points one builds simplicial complexes at several scales, and the homology groups, computed by exactly this matrix reduction, detect loops and voids in the data (Edelsbrunner and Harer, in the sources). The grid construction glues the top edge to the bottom edge with a flip ($j\mapsto n-j$) for the Klein bottle; without the flip it gives the torus again, as the second torus line confirms.